In [ ]:
!pip install -q --force-reinstall "numpy<2"

In [ ]:
!pip install -q --only-binary=:all: "pyarrow==15.0.2"

In [ ]:
!pip install -q "datasets==3.5.0"

In [ ]:
!pip install -q \
"transformers==4.52.4" \
"accelerate==1.7.0" \
"peft==0.15.2" \
"trl==0.18.1" \
"sentencepiece==0.1.99" \
einops \
scipy \
safetensors

In [ ]:
!pip install -q bitsandbytes

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("ALL GOOD")

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="automotive_en_dataset.jsonl"
)

dataset

In [ ]:
small_dataset = dataset["train"].shuffle(seed=42).select(range(4000))

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def format_chat(example):

    human = example["conversations"][0]["value"]
    assistant = example["conversations"][1]["value"]

    messages = [
        {
            "role": "system",
            "content": "You are an automotive expert assistant."
        },
        {
            "role": "user",
            "content": human
        },
        {
            "role": "assistant",
            "content": assistant
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [ ]:
small_dataset = small_dataset.map(format_chat)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./qwen3b-automotive",

    num_train_epochs=1,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    learning_rate=5e-5,

    logging_steps=10,

    save_strategy="no",

    bf16=True,

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    max_seq_length=512,

    packing=False,

    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=small_dataset,
    args=training_args,
)

In [ ]:
trainer.train()

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
prompt = "Explain symptoms of a failing alternator."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

result = pipe(
    text,

    max_new_tokens=120,

    temperature=0.7,
    top_p=0.9,

    repetition_penalty=1.1,

    do_sample=True,

    return_full_text=False,

    eos_token_id=terminators
)

print(result[0]["generated_text"])

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
trainer.model.save_pretrained(
    "./qwen3b-automotive-lora"
)

tokenizer.save_pretrained(
    "./qwen3b-automotive-lora-4k"
)

In [ ]:
trainer.model.push_to_hub(
    "Nasim435/qwen3b-automotive-lora"
)

tokenizer.push_to_hub(
    "Nasim435/qwen3b-automotive-lora"
)

In [ ]:
merged_model = model.merge_and_unload()

In [ ]:
merged_model.push_to_hub(
    "Nasim435/Qwen-3B-Automotive-4000"
)